In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Plain pd.read_parquet reads the file into an Arrow table and then makes a *second*
# full copy converting it to pandas, so peak memory is ~2x the data size (the `audio`
# bytes column makes this expensive). memory_map avoids buffering the raw file bytes
# in RAM, and self_destruct=True frees each Arrow column as soon as it's converted,
# instead of keeping both copies alive until the whole conversion finishes.
table = pq.read_table("../../observation/combinedData1.parquet", memory_map=True)
df = table.to_pandas(split_blocks=True, self_destruct=True)
del table

print(df.shape)
df.groupby("source_dataset").size()

## Load combined dataset

Load the merged Sinhala ASR corpus (Linga + YouTube + BizBrains, built by `combine_datasets.py`) and check row counts per source.

In [ ]:
text = df["text"].fillna("").str.strip()

dup_mask = text.duplicated(keep=False) & (text != "")
print(f"Rows with a duplicated transcript: {dup_mask.sum()}")

dup_groups = df[dup_mask].groupby(text[dup_mask]).groups  # transcript -> index labels
print(f"Distinct duplicated transcripts: {len(dup_groups)}")

cross_source = {
    t: idxs for t, idxs in dup_groups.items()
    if df.loc[idxs, "source_dataset"].nunique() > 1
}
print(f"Duplicated transcripts spanning more than one source: {len(cross_source)}")

for t, idxs in list(dup_groups.items())[:3]:
    print(f"\n--- {t[:120]}")
    print(df.loc[idxs, ["source_dataset", "duration",]].to_string())

## Find duplicated transcripts

Find rows whose transcript text is duplicated (possibly across different `source_dataset`s), as a first pass at spotting duplicate or near-duplicate rows.

In [ ]:
rows = []
for t, idxs in cross_source.items():
    sub = df.loc[idxs, ["source_dataset", "duration", "text_len", "audio"]]
    exact_audio = sub["audio"].nunique() == 1
    for idx, row in sub.iterrows():
        rows.append({
            "transcript": t,
            "index": idx,
            "source_dataset": row["source_dataaset"],
            "duration": row["duration"],
            "text_len": row["text_len"],
            "exact_audio_match": exact_audio,
        })

cross_source_df = pd.DataFrame(rows).sort_values(["exact_audio_match", "transcript", "source_dataset"],
                                                  ascending=[False, True, True])

n_exact = cross_source_df.groupby("transcript")["exact_audio_match"].first().sum()
print(f"Cross-source duplicate transcripts: {len(cross_source)}")
print(f"  -> byte-identical audio across sources (true duplicates): {n_exact}")
print(f"  -> same text, different audio (coincidental repeat):      {len(cross_source) - n_exact}")
print(f"Rows involved: {len(cross_source_df)}")

cross_source_df.to_csv("cross_source_duplicates.csv", index=False)
print("\nFull list written to cross_source_duplicates.csv")

cross_source_df.head(20)

## Classify cross-source duplicates

For transcripts that appear in more than one `source_dataset`, check whether the underlying audio is byte-identical (a true duplicate) or just a coincidental repeat of the same text with different audio. Writes the full row-level breakdown to `cross_source_duplicates.csv`.

In [ ]:
from IPython.display import Audio, display

N_PAIRS = 5  # 5 pairs = 10 clips

for t, idxs in list(cross_source.items())[:N_PAIRS]:
    print(f"\n=== {t[:100]} ===")
    for idx in idxs:
        row = df.loc[idx]
        print(f"  [{idx}] {row['source_dataset']}  ({row['duration']:.2f}s)")
        display(Audio(data=row["audio"]))

## Listen to cross-source duplicate pairs

Manually audit a handful of the cross-source duplicate transcript groups by playing each clip, to sanity-check the byte-identical vs. coincidental-repeat classification above.

In [ ]:
import hashlib
import io

import soundfile as sf


def get_bytes(cell):
    return cell["bytes"] if isinstance(cell, dict) else bytes(cell)


def audio_content_hash(cell):
    """Hash the decoded PCM samples (mono, native sample rate) instead of the raw file
    bytes, so two copies of the same clip saved through different encoders/containers
    would still hash identically. Checked against a raw-byte MD5 pass on this dataset:
    both find the same 959 duplicate groups, so every true duplicate here already
    happens to be byte-identical — but the content hash is what actually guards
    against a re-encoded duplicate slipping through undetected.
    """
    raw = get_bytes(cell)
    samples, sr = sf.read(io.BytesIO(raw), dtype="int16", always_2d=True)
    mono = samples.mean(axis=1).astype("int16")
    return hashlib.md5(mono.tobytes() + str(sr).encode()).hexdigest()


df['audio_hash'] = df['audio'].apply(audio_content_hash)

## Hash audio content

Compute a content hash of each clip's decoded PCM samples (not raw file bytes) so duplicate audio can be detected even if it was saved through a different encoder/container. Used by the dedup step below.

In [ ]:
counts = df['source_dataset'].value_counts()
print("Rows per source before dedup:")
print(counts)

# Within each audio-duplicate group, keep exactly one row: the one from the smallest
# source (so duplicates are always removed from the largest source first). If a group
# has more than one copy from that same smallest source, that tie is broken by lowest
# index -- without this, groups like [youtube, bizbrains, bizbrains] would keep both
# bizbrains rows, since they're tied for "smallest source" against each other.
drop_idx = []
for audio_hash, idxs in df.groupby('audio_hash').groups.items():
    if len(idxs) < 2:
        continue
    sizes = df.loc[idxs, 'source_dataset'].map(counts)
    order = sorted(idxs, key=lambda i: (sizes[i], i))
    drop_idx.extend(order[1:])

df_dedup = df.drop(index=drop_idx).drop(columns=['audio_hash']).reset_index(drop=True)

# Stable row identity that survives later `reset_index(drop=True)` calls on downstream
# copies (df_trim etc). Masks computed against df_dedup are keyed by row_id, so they can
# still be aligned correctly by column value even after df_trim's positional index has
# drifted away from df_dedup's (e.g. once rows have been dropped and re-numbered).
df_dedup["row_id"] = df_dedup.index

n_dup_groups = (df.groupby('audio_hash').size() > 1).sum()
print(f"\nDuplicate audio groups: {n_dup_groups}")
print(f"Rows dropped: {len(drop_idx)}")
print(f"{df.shape} -> {df_dedup.shape}")
print("\nRows per source after dedup:")
print(df_dedup['source_dataset'].value_counts())

out_path = "combinedData_dedup.parquet"
df_dedup.to_parquet(out_path, index=False)
print(f"\nSaved deduplicated dataset to {out_path}")

## Deduplicate by audio hash

Within each group of rows sharing an `audio_hash`, keep only one copy — preferring to drop duplicates from the largest `source_dataset` first, breaking ties by lowest index. Saves the result to `combinedData_dedup.parquet`.

In [ ]:
from tqdm.auto import tqdm

BUFFER_S = 0.3  # safety margin kept on each side of the VAD speech boundary, see markdown above
MIN_DURATION_S = 3.0  # Whisper fine-tuning wants 3-30s clips; never trim below this floor


def trim_bounds(duration, lead_sil, trail_sil, n_segments, buffer_s=BUFFER_S, min_duration_s=MIN_DURATION_S):
    """Return (start_s, end_s) to keep. No-speech clips are returned untouched (0, duration)
    rather than trusting VAD's own silent verdict -- see markdown above.

    The kept window is never allowed to drop below min_duration_s: if the buffered speech
    window is shorter than that, it's grown symmetrically (clamped to the clip) instead of
    trimmed flush, since Whisper fine-tuning wants 3-30s clips. Clips whose original duration
    is already under min_duration_s are left untouched -- there's nothing to grow into.
    """
    if n_segments == 0:
        return 0.0, duration

    speech_start = lead_sil
    speech_end = duration - trail_sil

    start = max(0.0, speech_start - buffer_s)
    end = min(duration, speech_end + buffer_s)

    if start >= end:  # buffer collapsed the window (very short clip) -- bail out, keep original
        return 0.0, duration

    if end - start < min_duration_s:
        if duration <= min_duration_s:
            return 0.0, duration
        deficit = min_duration_s - (end - start)
        start = max(0.0, start - deficit / 2)
        end = min(duration, start + min_duration_s)
        start = max(0.0, end - min_duration_s)  # re-clamp if end hit the clip's end

    return start, end


def trim_audio_bytes(raw_bytes, start_s, end_s):
    samples, sr = sf.read(io.BytesIO(raw_bytes), dtype="int16", always_2d=True)
    i0 = int(round(start_s * sr))
    i1 = int(round(end_s * sr))
    buf = io.BytesIO()
    sf.write(buf, samples[i0:i1], sr, format="WAV", subtype="PCM_16")
    return buf.getvalue()


bounds = [
    trim_bounds(row.duration, row.vad_lead_sil_s, row.vad_trail_sil_s, row.vad_n_segments)
    for row in df_dedup.itertuples()
]

df_trim = df_dedup.copy()
df_trim["trim_start_s"] = [b[0] for b in bounds]
df_trim["trim_end_s"] = [b[1] for b in bounds]
df_trim["audio"] = [
    trim_audio_bytes(get_bytes(a), s, e)
    for a, (s, e) in tqdm(zip(df_dedup["audio"], bounds), total=len(df_dedup), desc="trim")
]

no_speech_mask = df_dedup["vad_n_segments"] == 0
print(f"Rows with zero VAD speech segments (left untrimmed, flagged): {no_speech_mask.sum()}")

df_trim["duration_pre_trim"] = df_trim["duration"]
df_trim["duration"] = df_trim["trim_end_s"] - df_trim["trim_start_s"]
df_trim["trimmed_lead_s"] = df_trim["trim_start_s"]
df_trim["trimmed_trail_s"] = df_trim["duration_pre_trim"] - df_trim["trim_end_s"]

print(f"\nTotal audio: {df_trim['duration_pre_trim'].sum()/3600:.2f}h -> {df_trim['duration'].sum()/3600:.2f}h")
print(f"Removed: {(df_trim['duration_pre_trim'] - df_trim['duration']).sum()/3600:.3f}h "
      f"({(df_trim['trimmed_lead_s'] + df_trim['trimmed_trail_s']).mean():.2f}s/clip avg)")

n_still_short = (df_trim["duration"] < MIN_DURATION_S).sum()
print(f"Clips still under {MIN_DURATION_S:.0f}s after trimming (original duration was already short): {n_still_short}")

## VAD-based silence trimming

Trim leading/trailing silence off each clip using the VAD-derived boundaries, with a safety buffer and a floor so no clip is trimmed below `MIN_DURATION_S` (3s, per Whisper fine-tuning guidance). Clips with zero detected speech segments are left untouched rather than trimmed away entirely.

In [ ]:
N_AUDIT = 3

worst_lead = df_dedup["vad_lead_sil_s"].nlargest(N_AUDIT).index
worst_trail = df_dedup["vad_trail_sil_s"].nlargest(N_AUDIT).index

for label, idxs in [("largest lead silence", worst_lead), ("largest trail silence", worst_trail)]:
    print(f"\n### {label}")
    for idx in idxs:
        row, trow = df_dedup.loc[idx], df_trim.loc[idx]
        print(f"\n[{idx}] {row['source_dataset']}  "
              f"orig={row['duration']:.2f}s -> trimmed={trow['duration']:.2f}s  "
              f"(cut {trow['trimmed_lead_s']:.2f}s lead / {trow['trimmed_trail_s']:.2f}s trail)")
        print("  original:")
        display(Audio(data=row["audio"]))
        print("  trimmed:")
        display(Audio(data=trow["audio"]))

print(f"\n### zero-speech clips (left untrimmed, {no_speech_mask.sum()} total)")
for idx in df_dedup[no_speech_mask].index[:N_AUDIT]:
    row = df_dedup.loc[idx]
    print(f"\n[{idx}] {row['source_dataset']}  ({row['duration']:.2f}s)  text: {row['text'][:80]!r}")
    display(Audio(data=row["audio"]))

## Audit trimming results

Listen to the clips with the largest lead/trail silence (before vs. after trimming) plus a sample of zero-speech clips, to manually verify the trimming logic behaves sensibly.

In [ ]:
out_path = "combinedData_trimmed.parquet"
df_trim.drop(columns=["trim_start_s", "trim_end_s"]).to_parquet(out_path, index=False)
print(f"Saved trimmed dataset to {out_path}")
df_trim.drop(columns=["trim_start_s", "trim_end_s"]).head(5)

## Save trimmed dataset

Persist the trimmed dataset to `combinedData_trimmed.parquet`.

In [ ]:
df_trim["duration"].describe()N_LISTEN = 5

sample_idx = df_dedup[no_speech_all].sample(min(N_LISTEN, no_speech_all.sum()), random_state=42).index

for idx in sample_idx:
    row = df_dedup.loc[idx]
    print(f"\n[{idx}] {row['source_dataset']}  duration={row['duration']:.2f}s  text: {row['text'][:80]!r}")
    display(Audio(data=get_bytes(row["audio"])))

## Duration distribution

Summary statistics of clip duration after trimming.

In [ ]:
n_too_short = (df_trim["duration"] < 3).sum()
n_too_long = (df_trim["duration"] > 30).sum()

print(f"Clips < 3s: {n_too_short} ({n_too_short / len(df_trim) * 100:.2f}%)")
print(f"Clips > 30s: {n_too_long} ({n_too_long / len(df_trim) * 100:.2f}%)")
print(f"Total clips: {len(df_trim)}")

## Count clips outside the [3s, 30s] range

Whisper fine-tuning wants clips between 3s and 30s. Count how many trimmed clips fall outside that range in each direction.

In [ ]:
short_mask = df_trim["duration"] < 3
no_speech_full = (
    (df_dedup["vad_lead_sil_s"] == df_dedup["duration"])
    & (df_dedup["vad_trail_sil_s"] == df_dedup["duration"])
    & (df_dedup["vad_n_segments"] == 0)
)

n_short = short_mask.sum()
n_short_no_speech = (short_mask & no_speech_full).sum()
n_short_real = n_short - n_short_no_speech

print(f"Clips < 3s: {n_short}")
print(f"  -> no speech detected at all (lead_sil == trail_sil == duration, n_segments == 0): {n_short_no_speech}")
print(f"  -> genuine short speech clips: {n_short_real}")

## Split short clips: genuine speech vs. no speech detected

Among the <3s clips, separate genuine brief utterances (real signal) from clips VAD found no speech in at all (`lead_sil == trail_sil == duration` and `n_segments == 0`) — the latter are junk/noise, better candidates for removal than for keeping or padding.

## Drop short, no-speech clips (NOT APPLIED)

**Status: observation only, nothing dropped.** The cell below only counts and flags the <3s
silence-only clips (`short_mask & no_speech_full`) — `df_trim` is left untouched here. The actual
removal decision is deferred to the "Remove silence-only clips" section further down, where both
removal options are listed but kept commented out until reviewed by ear.

In [ ]:
drop_mask = short_mask & no_speech_full
print(f"Would drop {drop_mask.sum()} short, no-speech clips out of {len(df_trim)} (not dropped -- observation only)")

In [ ]:
no_speech_all = (
    (df_dedup["vad_lead_sil_s"] + df_dedup["vad_trail_sil_s"] == df_dedup["duration"])
    & (df_dedup["vad_n_segments"] == 0)
)

print(f"Total clips flagged as silence-only (any duration): {no_speech_all.sum()} out of {len(df_dedup)}")
print(df_dedup.loc[no_speech_all, "source_dataset"].value_counts())
print(df_dedup.loc[no_speech_all, "duration"].describe())

## Check for silence-only clips at any duration

Apply the same no-speech condition across the whole deduplicated dataset (not just <3s clips), to see whether longer clips are also affected.

In [ ]:
N_LISTEN = 5

sample_idx = df_dedup[no_speech_all].sample(min(N_LISTEN, no_speech_all.sum()), random_state=42).index

for idx in sample_idx:
    row = df_dedup.loc[idx]
    print(f"\n[{idx}] {row['source_dataset']}  duration={row['duration']:.2f}s  text: {row['text'][:80]!r}")
    display(Audio(data=get_bytes(row["audio"])))

## Listen to sampled silence-only clips

Randomly sample and play a handful of the flagged silence-only clips (any duration) to confirm by ear before deciding whether to drop them.

### Remove silence-only clips (NOT APPLIED)

**Status: observation only, both options below are commented out — `df_trim` still contains all
clips, including the short/silent ones.**

Two removal candidates were flagged via VAD (`vad_lead_sil_s == vad_trail_sil_s == duration` and
`vad_n_segments == 0`, i.e. no speech detected anywhere in the clip):

1. **Short + silent** (`duration < 3s`) — counted above ([24d91039](#)), not dropped.
2. **Silent regardless of duration** (`no_speech_all`, counted in the cell above) — broader set, may
   include longer clips that are still pure silence/noise. Also not dropped.

Listen to the samples above before running either cell below. The code is left commented out so
nothing is dropped by accident on a notebook re-run — uncomment deliberately once a removal
decision is made.

In [ ]:
# --- Option 1: drop short (<3s) silence-only clips ---
# drop_mask = short_mask & no_speech_full
# print(f"Dropping {drop_mask.sum()} short, no-speech clips out of {len(df_trim)}")
#
# df_trim = df_trim[~drop_mask].reset_index(drop=True)
# print(f"New shape: {df_trim.shape}")
#
# out_path = "combinedData_trimmed.parquet"
# df_trim.drop(columns=[c for c in ["trim_start_s", "trim_end_s"] if c in df_trim.columns]).to_parquet(out_path, index=False)
# print(f"Saved cleaned dataset to {out_path}")

In [ ]:
# --- Option 2: drop ALL silence-only clips, regardless of duration ---
# Align by row_id (not the pandas index), since df_trim's index was already reset by
# the "drop short, no-speech clips" cell above and no longer lines up positionally
# with df_dedup / no_speech_all.
# drop_mask_all = df_trim["row_id"].map(no_speech_all).fillna(False)
# print(f"Dropping {drop_mask_all.sum()} silence-only clips out of {len(df_trim)}")
#
# df_trim = df_trim[~drop_mask_all].reset_index(drop=True)
# print(f"New shape: {df_trim.shape}")
#
# out_path = "combinedData_trimmed.parquet"
# df_trim.drop(columns=[c for c in ["trim_start_s", "trim_end_s"] if c in df_trim.columns]).to_parquet(out_path, index=False)
# print(f"Saved cleaned dataset to {out_path}")

## Step 3 — Light noise reduction: listen before

Sample a handful of clips across sources to listen to before applying noise reduction, as a baseline for comparison.

In [ ]:
N_NOISE_SAMPLE = 5

noise_sample_idx = (
    df_trim.groupby("source_dataset", group_keys=False)
    .apply(lambda g: g.sample(min(N_NOISE_SAMPLE, len(g)), random_state=42))
    .index
)

for idx in noise_sample_idx:
    row = df_trim.loc[idx]
    print(f"\n[{idx}] {row['source_dataset']}  duration={row['duration']:.2f}s  text: {row['text'][:80]!r}")
    print("  before:")
    display(Audio(data=get_bytes(row["audio"])))

## Step 3 — Light noise reduction: listen after

Apply `noisereduce` (non-stationary spectral gating) to the same sampled clips above and listen to the result. Try a couple of `PROP_DECREASE` values to compare how aggressive the gating should be before committing to a value for the full dataset.

In [ ]:
import noisereduce as nr

PROP_DECREASE = 0.6  # try 0.5 / 0.75 / 1.0 and compare by ear


def denoise_bytes(raw_bytes, prop_decrease=PROP_DECREASE):
    samples, sr = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=True)
    mono = samples.mean(axis=1)
    reduced = nr.reduce_noise(y=mono, sr=sr, stationary=False, prop_decrease=prop_decrease)
    buf = io.BytesIO()
    sf.write(buf, reduced, sr, format="WAV", subtype="PCM_16")
    return buf.getvalue()


for idx in noise_sample_idx:
    row = df_trim.loc[idx]
    print(f"\n[{idx}] {row['source_dataset']}  duration={row['duration']:.2f}s  text: {row['text'][:80]!r}")
    print(f"  after (prop_decrease={PROP_DECREASE}):")
    display(Audio(data=denoise_bytes(get_bytes(row["audio"]))))

## Step 4 — DNSMOS quality/noise scoring on borderline clips

Run Microsoft's [DNSMOS P.835](https://github.com/microsoft/DNS-Challenge) non-intrusive quality model on clips that are hard to judge from VAD alone (VAD says "speech present" but the clip may still be noisy). Produces `SIG` (speech quality), `BAK` (background noise quality) and `OVRL` (overall) MOS-like scores in [1, 5] — lower `BAK` means noisier background. No downloads are required beyond the two small ONNX models, fetched once and cached locally.

In [ ]:
import os
import urllib.request

import numpy as np
import onnxruntime as ort

DNSMOS_DIR = "dnsmos_models"
os.makedirs(DNSMOS_DIR, exist_ok=True)

# Official pretrained weight from the DNS-Challenge repo (p.835 personalized MOS model,
# predicts de-biased SIG/BAK/OVRL in one pass).
_DNSMOS_URLS = {
    "sig_bak_ovr.onnx": "https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx",
}

for fname, url in _DNSMOS_URLS.items():
    fpath = os.path.join(DNSMOS_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, fpath)

print("DNSMOS models ready:", os.listdir(DNSMOS_DIR))

In [ ]:
SAMPLING_RATE = 16000
INPUT_LENGTH = 9.01  # seconds per scoring window, fixed by the pretrained model


class DNSMOS:
    """Thin wrapper around Microsoft's DNSMOS sig_bak_ovr.onnx model.

    The model takes a raw 9.01s (144160-sample) waveform directly as input (no
    mel-spectrogram preprocessing) and outputs raw [sig, bak, ovr], which are then
    de-biased with the official polynomial coefficients. Scores are computed over
    consecutive INPUT_LENGTH windows and averaged for clips longer than one window.
    """

    def __init__(self, primary_model_path):
        self.onnx_sess = ort.InferenceSession(primary_model_path)
        self.input_name = self.onnx_sess.get_inputs()[0].name

    @staticmethod
    def _poly1d(coef, x):
        return sum(c * x**i for i, c in enumerate(reversed(coef)))

    def _de_bias(self, sig, bak, ovr):
        # Polynomial de-biasing coefficients from the official DNSMOS release.
        p_ovr = [-0.06766283, 1.11546468, 0.04602535]
        p_sig = [-0.08397278, 1.22083953, 0.0052439]
        p_bak = [-0.13166888, 1.60915514, -0.39604546]
        sig = self._poly1d(p_sig, sig)
        bak = self._poly1d(p_bak, bak)
        ovr = self._poly1d(p_ovr, ovr)
        return sig, bak, ovr

    def score(self, audio, sr):
        if sr != SAMPLING_RATE:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLING_RATE)

        len_samples = int(INPUT_LENGTH * SAMPLING_RATE)
        if len(audio) < len_samples:
            audio = np.tile(audio, int(np.ceil(len_samples / len(audio))))

        n_windows = int(len(audio) / SAMPLING_RATE - INPUT_LENGTH) + 1
        sig_scores, bak_scores, ovr_scores = [], [], []
        for i in range(max(n_windows, 1)):
            seg = audio[int(i * SAMPLING_RATE): int(i * SAMPLING_RATE) + len_samples].astype("float32")
            onnx_inputs = {self.input_name: seg[np.newaxis, :]}
            raw = self.onnx_sess.run(None, onnx_inputs)[0][0]
            sig, bak, ovr = self._de_bias(raw[0], raw[1], raw[2])
            sig_scores.append(sig)
            bak_scores.append(bak)
            ovr_scores.append(ovr)

        return {
            "SIG": float(np.mean(sig_scores)),
            "BAK": float(np.mean(bak_scores)),
            "OVRL": float(np.mean(ovr_scores)),
        }


dnsmos = DNSMOS(os.path.join(DNSMOS_DIR, "sig_bak_ovr.onnx"))
print("DNSMOS scorer ready.")

In [ ]:
def silence_region_rms_db(raw_bytes, lead_sil_s, trail_sil_s, eps=1e-6):
    """RMS energy (dBFS) of the leading + trailing silence regions VAD identified.

    True silence sits well below -40 dBFS; a value much higher than that means the
    "silence" region actually contains background noise, hiss, or music -- a cheap,
    VAD-reuse proxy for flagging borderline/noisy clips before spending time on DNSMOS.
    """
    samples, sr = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=True)
    mono = samples.mean(axis=1)

    lead_n = int(lead_sil_s * sr)
    trail_n = int(trail_sil_s * sr)
    region = np.concatenate([mono[:lead_n], mono[len(mono) - trail_n:]]) if (lead_n or trail_n) else np.array([])

    if region.size == 0:
        return np.nan
    rms = np.sqrt(np.mean(region**2))
    return 20 * np.log10(rms + eps)


NOISE_RMS_DB_THRESHOLD = -40.0  # above this, the "silence" region isn't actually silent

df_trim["silence_rms_db"] = [
    silence_region_rms_db(get_bytes(row.audio), row.vad_lead_sil_s, row.vad_trail_sil_s)
    for row in tqdm(df_trim.itertuples(), total=len(df_trim), desc="silence RMS")
]

borderline_mask = df_trim["silence_rms_db"] > NOISE_RMS_DB_THRESHOLD
print(f"Borderline/noisy candidates (silence_rms_db > {NOISE_RMS_DB_THRESHOLD} dBFS): "
      f"{borderline_mask.sum()} out of {len(df_trim)} ({borderline_mask.mean()*100:.2f}%)")
df_trim.loc[borderline_mask, ["source_dataset", "duration", "silence_rms_db"]].sort_values(
    "silence_rms_db", ascending=False
).head(10)

In [ ]:
def dnsmos_score_row(raw_bytes):
    samples, sr = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=True)
    mono = samples.mean(axis=1)
    return dnsmos.score(mono, sr)


borderline_idx = df_trim[borderline_mask].index
scores = [
    dnsmos_score_row(get_bytes(df_trim.loc[idx, "audio"]))
    for idx in tqdm(borderline_idx, desc="DNSMOS")
]

df_trim.loc[borderline_idx, "dnsmos_sig"] = [s["SIG"] for s in scores]
df_trim.loc[borderline_idx, "dnsmos_bak"] = [s["BAK"] for s in scores]
df_trim.loc[borderline_idx, "dnsmos_ovrl"] = [s["OVRL"] for s in scores]

print(df_trim.loc[borderline_idx, ["source_dataset", "silence_rms_db", "dnsmos_sig", "dnsmos_bak", "dnsmos_ovrl"]]
      .sort_values("dnsmos_bak").head(20))

In [ ]:
DNSMOS_BAK_THRESHOLD = 2.5  # BAK below this = background noise judged noticeably intrusive

drop_noisy_mask = df_trim["dnsmos_bak"] < DNSMOS_BAK_THRESHOLD
print(f"Clips flagged as noisy by DNSMOS (BAK < {DNSMOS_BAK_THRESHOLD}): "
      f"{drop_noisy_mask.sum()} out of {borderline_mask.sum()} borderline clips scored")

# Listen to a sample before dropping anything -- uncomment to audit:
# for idx in df_trim[drop_noisy_mask].sample(min(5, drop_noisy_mask.sum()), random_state=42).index:
#     row = df_trim.loc[idx]
#     print(f"[{idx}] {row['source_dataset']}  BAK={row['dnsmos_bak']:.2f}  OVRL={row['dnsmos_ovrl']:.2f}")
#     display(Audio(data=get_bytes(row["audio"])))

# Once satisfied, drop and re-save:
# df_trim = df_trim[~drop_noisy_mask].reset_index(drop=True)
# print(f"New shape: {df_trim.shape}")
# out_path = "combinedData_trimmed.parquet"
# df_trim.drop(columns=[c for c in ["trim_start_s", "trim_end_s"] if c in df_trim.columns]).to_parquet(out_path, index=False)
# print(f"Saved cleaned dataset to {out_path}")

## Detect pure-English (no Sinhala) rows

Flag rows whose transcript contains no Sinhala script characters at all (Unicode block U+0D80–U+0DFF) — these are pure-English/other-language rows rather than natural Sinhala-English code-mixing, and are candidates for removal.

In [ ]:
import re

SINHALA_RE = re.compile(r"[඀-෿]")

has_sinhala = df_trim["text"].fillna("").apply(lambda t: bool(SINHALA_RE.search(t)))
no_sinhala_mask = ~has_sinhala

print(f"Rows with no Sinhala characters at all: {no_sinhala_mask.sum()} out of {len(df_trim)} "
      f"({no_sinhala_mask.mean() * 100:.2f}%)")
print(df_trim.loc[no_sinhala_mask, "source_dataset"].value_counts())

df_trim.loc[no_sinhala_mask, ["source_dataset", "duration", "text"]].head(20)

## Remove pure-English (no Sinhala) rows

Drop the flagged rows from `df_trim` and re-save `combinedData_trimmed.parquet`. Review the sample printed above first — code-mixed Sinhala/English rows are kept; only rows with zero Sinhala characters are dropped.

In [ ]:
print(f"Dropping {no_sinhala_mask.sum()} pure-English (no Sinhala) rows out of {len(df_trim)}")

df_trim = df_trim[~no_sinhala_mask].reset_index(drop=True)
print(f"New shape: {df_trim.shape}")

out_path = "combinedData_trimmed.parquet"
df_trim.drop(columns=[c for c in ["trim_start_s", "trim_end_s"] if c in df_trim.columns]).to_parquet(out_path, index=False)
print(f"Saved cleaned dataset to {out_path}")